# Example 3 — AI Pictionary
**Classify 10 kinds of doodles, then turn the CNN into a drawing game**  
**Difficulty:** Intermediate

A convolutional neural network will learn patterns from thousands of small doodles. After training, you will photograph or upload your own drawing and let the CNN make its top three guesses.

The ten categories are:

- apple
- bicycle
- cat
- fish
- flower
- house
- ice cream
- star
- tree
- umbrella

This is a **closed-set classifier**. It must choose among these ten categories, even when the uploaded drawing does not belong to any of them.

### Your mission

1. Explore the doodle data.
2. Normalize the pixels.
3. Design and train a CNN.
4. evaluate it on unseen drawings.
5. inspect predictions, a confusion matrix, and Grad-CAM.
6. preprocess a photograph into the CNN's 28 × 28 format.
7. turn the classifier into a playable Pictionary game.

## Colab use
Run cells from top to bottom. CPU works; a free GPU is optional. Downloaded data and uploaded drawings are cached only for the current Colab runtime. If the session disconnects, rerun earlier cells.

The instructor should configure the dataset location once in the data-loading cell. Students should not need to edit that setup.

In [1]:
# Free Google Colab CPU is sufficient. A free GPU is optional.

import random
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from PIL import Image, ImageOps

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
)

keras.utils.set_random_seed(42)
random.seed(42)

print('TensorFlow:', tf.__version__)
print('Target image shape: (28, 28, 1)')
print('Devices:', tf.config.list_physical_devices())

TensorFlow: 2.20.0
Target image shape: (28, 28, 1)
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [ ]:
def ensure_grayscale_channel(images):
    """Return uint8 images with shape (examples, 28, 28, 1)."""
    images = np.asarray(images)

    if images.ndim == 3:
        images = images[..., np.newaxis]

    if images.ndim != 4 or images.shape[-1] != 1:
        raise ValueError(
            'Expected grayscale images shaped (N, 28, 28) '
            'or (N, 28, 28, 1).'
        )

    if images.shape[1:3] != (28, 28):
        raise ValueError(
            f'Expected 28x28 images, received {images.shape[1:3]}.'
        )

    return images.astype(np.uint8)


def load_pictionary_npz(dataset_path):
    """
    Load the prepared AI Pictionary dataset.

    Supported formats:
      1. X_train, y_train, X_validation, y_validation, X_test, y_test
      2. X_train, y_train, X_test, y_test
      3. x_train_raw, y_train, x_test_raw, y_test

    The first format is used by the original camp dataset. Its training and
    validation arrays are combined so this lab can use validation_split,
    matching the other CNN notebooks.
    """
    dataset_path = Path(dataset_path)

    if not dataset_path.exists():
        raise FileNotFoundError(
            'The dataset file was not found.\n'
            f'Expected location: {dataset_path}\n\n'
            'Ask the instructor for the shared Drive path or direct URL.'
        )

    with np.load(dataset_path, allow_pickle=False) as data:
        available = set(data.files)

        if {
            'X_train', 'y_train',
            'X_validation', 'y_validation',
            'X_test', 'y_test'
        }.issubset(available):
            x_train_raw = np.concatenate(
                [data['X_train'], data['X_validation']],
                axis=0
            )
            y_train = np.concatenate(
                [data['y_train'], data['y_validation']],
                axis=0
            )
            x_test_raw = data['X_test']
            y_test = data['y_test']

        elif {
            'X_train', 'y_train',
            'X_test', 'y_test'
        }.issubset(available):
            x_train_raw = data['X_train']
            y_train = data['y_train']
            x_test_raw = data['X_test']
            y_test = data['y_test']

        elif {
            'x_train_raw', 'y_train',
            'x_test_raw', 'y_test'
        }.issubset(available):
            x_train_raw = data['x_train_raw']
            y_train = data['y_train']
            x_test_raw = data['x_test_raw']
            y_test = data['y_test']

        else:
            raise KeyError(
                'The NPZ file does not contain a recognized set of arrays. '
                f'Available keys: {sorted(available)}'
            )

        if 'class_names' in available:
            class_names = data['class_names'].astype(str).tolist()
        else:
            class_names = [
                'apple',
                'bicycle',
                'cat',
                'fish',
                'flower',
                'house',
                'ice cream',
                'star',
                'tree',
                'umbrella',
            ]

    x_train_raw = ensure_grayscale_channel(x_train_raw)
    x_test_raw = ensure_grayscale_channel(x_test_raw)
    y_train = np.asarray(y_train, dtype=np.int64)
    y_test = np.asarray(y_test, dtype=np.int64)

    return (
        x_train_raw,
        x_test_raw,
        y_train,
        y_test,
        class_names,
    )

## Load the dataset
The prepared dataset is stored as one compressed `.npz` file. The images remain `uint8` until the student normalization step.

The instructor can use either:

- a direct public link to the `.npz` file, or
- a shared file in Google Drive.

Only the configuration values at the top of the next cell should need editing.

In [3]:
# INSTRUCTOR SETUP
# Option A: paste a direct-download URL to the prepared NPZ file.
DATA_URL = ''

# Option B: keep DATA_URL empty and edit the shared Google Drive path.
DRIVE_DATA_FILE = (
    '/content/drive/MyDrive/AI Summer Camp/data/'
    'ai_pictionary_quickdraw.npz'
)


if DATA_URL.strip():
    dataset_path = Path(
        keras.utils.get_file(
            fname='ai_pictionary_quickdraw.npz',
            origin=DATA_URL,
        )
    )
    print('Using direct dataset download.')

else:
    try:
        from google.colab import drive
    except ModuleNotFoundError as error:
        raise RuntimeError(
            'This Drive-loading option must run in Google Colab. '
            'Alternatively, set DATA_URL to a direct-download link.'
        ) from error

    drive.mount('/content/drive')
    dataset_path = Path(DRIVE_DATA_FILE)
    print('Using the dataset stored in Google Drive.')


(
    x_train_raw,
    x_test_raw,
    y_train,
    y_test,
    class_names,
) = load_pictionary_npz(dataset_path)

input_shape = (28, 28, 1)
num_classes = len(class_names)

print('Dataset file:', dataset_path)

ValueError: mount failed

In [ ]:
print('Training images:', x_train_raw.shape)
print('Test images:', x_test_raw.shape)
print('Raw dtype:', x_train_raw.dtype)
print('Raw range:', int(x_train_raw.min()), 'to', int(x_train_raw.max()))
print('Classes:', class_names)
print('Training class counts:', np.bincount(y_train, minlength=num_classes))
print('Test class counts:', np.bincount(y_test, minlength=num_classes))

## Explore first
Random guessing gives about **10%** accuracy. Look for:

- the strokes or shapes that distinguish each category,
- categories that may look similar,
- variation in drawing style,
- incomplete or unusual doodles,
- possible shortcuts the CNN could learn.

In [ ]:
def show_examples(images, labels, class_names, rows=3, cols=5):
    total = min(rows * cols, len(images))
    chosen = np.random.choice(
        len(images),
        size=total,
        replace=False
    )

    plt.figure(figsize=(12, 7))

    for plot_position, image_index in enumerate(chosen, start=1):
        plt.subplot(rows, cols, plot_position)
        plt.imshow(
            np.squeeze(images[image_index]),
            cmap='gray',
            vmin=0,
            vmax=255
        )
        plt.title(class_names[int(labels[image_index])])
        plt.axis('off')

    plt.tight_layout()
    plt.show()


show_examples(x_train_raw, y_train, class_names)

## Mathematics in friendly language

Every image has shape

$
28\times28\times1,
$

so it contains $28\cdot28\cdot1=784$ pixel values.

### Normalization

$
x_{\text{normalized}}=\frac{x_{\text{raw}}}{255}.
$

The doodle does not visually change. Only its numerical scale changes from 0–255 to 0–1.

### Convolution

A filter computes a local weighted sum:

$
z_{i,j,k}=b_k+\sum_{u,v,c}W_{u,v,c,k}X_{i+u,j+v,c}.
$

Early filters may detect short lines, curves, corners, and intersections. Deeper filters combine them into larger object parts.

### ReLU and pooling

$
\operatorname{ReLU}(z)=\max(0,z).
$

A $2\times2$ max-pooling operation can reduce $28\times28$ to $14\times14$, making later computation cheaper.

### Softmax and loss

$
p_k=\frac{e^{s_k}}{\sum_j e^{s_j}},
\qquad
\mathcal{L}=-\log p_y.
$

Softmax probabilities add to 1. Cross-entropy is small for a confident correct answer and large for a confident wrong answer.

### Learning

$
\theta\leftarrow\theta-\eta\nabla_\theta\mathcal{L}.
$

The gradient indicates how the loss changes, and the learning rate $\eta$ controls the update size.

## STUDENT TODO 1 — Normalize
Create `x_train` and `x_test`. Keep labels as integer class IDs.

In [ ]:
# STUDENT TODO 1: NORMALIZE THE PIXELS
# Create float32 arrays x_train and x_test with values from 0 to 1.
# Mathematical hint: normalized_pixel = raw_pixel / 255.0

x_train = None
x_test = None

print('Replace None with your normalization code.')

In [ ]:
def check_normalization(x_train, x_test):
    if x_train is None or x_test is None:
        print('Normalization is not complete.')
        return False

    ok = (
        x_train.shape[1:] == (28, 28, 1)
        and x_test.shape[1:] == (28, 28, 1)
        and x_train.dtype == np.float32
        and x_test.dtype == np.float32
        and 0 <= float(x_train.min()) <= float(x_train.max()) <= 1
        and 0 <= float(x_test.min()) <= float(x_test.max()) <= 1
    )

    print('Shape:', x_train.shape)
    print('dtype:', x_train.dtype)
    print('range:', float(x_train.min()), 'to', float(x_train.max()))
    print('Passed!' if ok else 'Check shape, dtype, and range again.')

    return ok


check_normalization(x_train, x_test)

## Optional augmentation
Augmentation creates small training-time transformations and can improve robustness. Test data must not be augmented.

For doodles, small translations, rotations, and zooms are reasonable. Horizontal flipping is omitted because it may change the meaning of some asymmetric drawings.

In [ ]:
# Optional: students may include this object near the start of their CNN.

data_augmentation = keras.Sequential([
    layers.RandomTranslation(
        height_factor=0.08,
        width_factor=0.08,
        fill_mode='constant',
        fill_value=0,
    ),
    layers.RandomRotation(
        0.05,
        fill_mode='constant',
        fill_value=0,
    ),
    layers.RandomZoom(
        height_factor=(-0.08, 0.08),
        width_factor=(-0.08, 0.08),
        fill_mode='constant',
        fill_value=0,
    ),
], name='data_augmentation')

print('Optional augmentation block is ready.')

## STUDENT TODO 2 — Design the CNN
Try two convolution blocks and remain under 500,000 parameters. Your architecture is a testable hypothesis, not a guaranteed answer.

In [ ]:
# STUDENT TODO 2: DEFINE AND COMPILE A CNN
# Required input shape: (28, 28, 1)
# Suggested budget: 2-3 Conv2D layers, 8-64 filters, under 500K parameters.
# Useful layers:
#   layers.Input(shape=input_shape)
#   data_augmentation
#   layers.Conv2D(16, 3, padding='same', activation='relu')
#   layers.MaxPooling2D(2)
#   layers.Dropout(0.2)
#   layers.GlobalAveragePooling2D()
#   layers.Flatten()
#   layers.Dense(32, activation='relu')
#   layers.Dense(num_classes, activation='softmax')
#
# Compile with Adam, sparse categorical cross-entropy, and accuracy.

model = None
print('Define and compile your model.')

In [ ]:
def check_model(model):
    if model is None:
        print('Model is not defined.')
        return False

    has_conv = any(
        isinstance(layer, layers.Conv2D)
        for layer in model.layers
    )
    correct_input = tuple(model.input_shape[1:]) == input_shape
    correct_output = model.output_shape[-1] == num_classes
    params = model.count_params()

    print('Has Conv2D:', has_conv)
    print('Input:', model.input_shape, 'Output:', model.output_shape)
    print('Parameters:', f'{params:,}')

    if params > 500_000:
        print(
            'Warning: use fewer filters, a smaller Dense layer, '
            'or GlobalAveragePooling2D.'
        )

    ok = has_conv and correct_input and correct_output

    if ok:
        model.summary()
    else:
        print('Revise the input, convolution, or final output layer.')

    return ok


check_model(model)

## STUDENT TODO 3 — Train
An epoch is one full pass through training data. Validation data measures progress without updating weights.

In [ ]:
# STUDENT TODO 3: TRAIN THE CNN
# Use model.fit(x_train, y_train, validation_split=0.20, ...)
# Suggested: 6-12 epochs and batch_size 32, 64, or 128.
# Save the returned object as history.

history = None
print('Train the model and save the result as history.')

## Interpret the curves
Training and validation improving together is healthy. A growing gap often indicates overfitting. Validation loss rising while training loss falls is another warning sign.

In [ ]:
def plot_learning_curves(history):
    if history is None:
        print('Train the model first.')
        return

    h = history.history
    epochs = range(1, len(h['loss']) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(
        epochs,
        h['accuracy'],
        marker='o',
        label='Training accuracy'
    )
    plt.plot(
        epochs,
        h['val_accuracy'],
        marker='o',
        label='Validation accuracy'
    )
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Accuracy curves')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(
        epochs,
        h['loss'],
        marker='o',
        label='Training loss'
    )
    plt.plot(
        epochs,
        h['val_loss'],
        marker='o',
        label='Validation loss'
    )
    plt.xlabel('Epoch')
    plt.ylabel('Cross-entropy loss')
    plt.title('Loss curves')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


plot_learning_curves(history)

## STUDENT TODO 4 — Test
Use the test set as a final exam, not as repeated feedback for redesigning the model.

In [ ]:
# STUDENT TODO 4: TEST THE CNN
# Use model.evaluate(x_test, y_test, verbose=0).
# Save the outputs as test_loss and test_accuracy, then print accuracy as a percent.

test_loss = None
test_accuracy = None

print('Evaluate the model on the test set.')

## Individual predictions
Softmax confidence is not certainty. A confident prediction can still be wrong.

In [ ]:
def show_predictions(model, images, labels, class_names, count=12):
    if model is None or images is None:
        print('Finish normalization and training first.')
        return

    count = min(count, len(images))
    probs = model.predict(images[:count], verbose=0)
    preds = np.argmax(probs, axis=1)

    cols = 4
    rows = int(np.ceil(count / cols))
    plt.figure(figsize=(12, 3.2 * rows))

    for i in range(count):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(
            np.squeeze(images[i]),
            cmap='gray',
            vmin=0,
            vmax=1
        )

        predicted = int(preds[i])
        true = int(labels[i])
        mark = '✓' if predicted == true else '✗'

        plt.title(
            f'{mark} Pred: {class_names[predicted]}\n'
            f'True: {class_names[true]}\n'
            f'Confidence: {100 * probs[i, predicted]:.1f}%'
        )
        plt.axis('off')

    plt.tight_layout()
    plt.show()


if x_test is None:
    print('Normalize the data first.')
else:
    show_predictions(model, x_test, y_test, class_names)

## Confusion matrix

Entry $C_{ij}$ counts drawings whose true class is $i$ and predicted class is $j$. Diagonal values are correct; off-diagonal values reveal specific mistakes.

In [ ]:
def plot_confusion(model, images, labels, class_names):
    if model is None or images is None:
        print('Finish normalization and training first.')
        return

    preds = np.argmax(
        model.predict(images, verbose=0),
        axis=1
    )
    matrix = confusion_matrix(labels, preds)

    size = max(8, 0.8 * len(class_names))
    fig, ax = plt.subplots(figsize=(size, size))

    ConfusionMatrixDisplay(
        matrix,
        display_labels=class_names
    ).plot(
        ax=ax,
        cmap='Blues',
        xticks_rotation=45,
        colorbar=False
    )

    plt.title('AI Pictionary Confusion Matrix')
    plt.tight_layout()
    plt.show()


if x_test is None:
    print('Normalize the data first.')
else:
    plot_confusion(model, x_test, y_test, class_names)

## Grad-CAM mathematics

Grad-CAM estimates which spatial locations supported class $c$. For feature map $A^k$,

$
\alpha_k^c=\frac{1}{Z}\sum_{i,j}\frac{\partial s^c}{\partial A_{i,j}^k},
$

and

$
L_{\text{Grad-CAM}}^c=\operatorname{ReLU}\left(\sum_k\alpha_k^c A^k\right).
$

Bright areas strongly supported the selected class. This is useful evidence, but it is not proof that the CNN understands the drawing like a human.

In [ ]:
def last_conv_layer(model):
    """Return the final Conv2D layer in a model."""
    for layer in reversed(model.layers):
        if isinstance(layer, layers.Conv2D):
            return layer

    raise ValueError('Grad-CAM requires at least one Conv2D layer.')


def build_gradcam_model(model, image_shape):
    """Create a functional model that exposes convolution features."""
    target_conv_layer = last_conv_layer(model)

    symbolic_input = keras.Input(
        shape=image_shape,
        name='gradcam_input'
    )
    x = symbolic_input
    convolution_features = None

    for layer in model.layers:
        try:
            x = layer(x, training=False)
        except TypeError:
            x = layer(x)

        if layer is target_conv_layer:
            convolution_features = x

    if convolution_features is None:
        raise ValueError(
            'The final Conv2D layer was not connected to the prediction.'
        )

    return keras.Model(
        inputs=symbolic_input,
        outputs=[convolution_features, x],
        name='gradcam_model'
    )


def make_gradcam_heatmap(image_batch, model, class_index=None):
    """Compute a normalized Grad-CAM heatmap for one image."""
    grad_model = build_gradcam_model(
        model,
        image_shape=tuple(image_batch.shape[1:])
    )

    with tf.GradientTape() as tape:
        features, predictions = grad_model(
            image_batch,
            training=False
        )

        if class_index is None:
            class_index = tf.argmax(predictions[0])

        selected_score = predictions[:, class_index]

    gradients = tape.gradient(selected_score, features)

    if gradients is None:
        raise RuntimeError(
            'Gradients could not be computed. Check that the final '
            'Conv2D layer is connected to the output.'
        )

    weights = tf.reduce_mean(
        gradients,
        axis=(0, 1, 2)
    )

    heatmap = tf.reduce_sum(
        features[0] * weights,
        axis=-1
    )
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (
        tf.reduce_max(heatmap) + keras.backend.epsilon()
    )

    return heatmap.numpy(), int(class_index)


def show_gradcam(model, image, class_names, true_label=None):
    """Show a doodle, its Grad-CAM heatmap, and an overlay."""
    if model is None:
        print('Define and train the model first.')
        return

    image_batch = np.expand_dims(
        image,
        axis=0
    ).astype('float32')

    probabilities = model.predict(
        image_batch,
        verbose=0
    )[0]
    predicted_index = int(np.argmax(probabilities))

    heatmap, _ = make_gradcam_heatmap(
        image_batch,
        model,
        class_index=predicted_index
    )

    resized_heatmap = tf.image.resize(
        heatmap[..., np.newaxis],
        image.shape[:2]
    ).numpy().squeeze()

    display_gray = np.squeeze(
        np.clip(image, 0, 1)
    )
    display_rgb = np.repeat(
        display_gray[..., np.newaxis],
        3,
        axis=-1
    )

    colored_heatmap = plt.cm.jet(
        resized_heatmap
    )[..., :3]

    overlay = np.clip(
        0.60 * display_rgb
        + 0.40 * colored_heatmap,
        0,
        1
    )

    title = (
        f'Predicted: {class_names[predicted_index]} '
        f'({100 * probabilities[predicted_index]:.1f}%)'
    )

    if true_label is not None:
        title += f' | True: {class_names[int(true_label)]}'

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(display_gray, cmap='gray', vmin=0, vmax=1)
    plt.title('Original doodle')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(resized_heatmap, cmap='jet')
    plt.title('Grad-CAM heatmap')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title(title)
    plt.axis('off')

    plt.tight_layout()
    plt.show()


gradcam_index = 0

if x_test is None:
    print('Normalize the images before using Grad-CAM.')
else:
    show_gradcam(
        model,
        x_test[gradcam_index],
        class_names,
        true_label=y_test[gradcam_index]
    )

# From Classifier to Game

The test images are clean 28 × 28 digital doodles:

- black background,
- bright drawing strokes,
- no paper texture, hand, desk, or shadow.

A photograph looks different. The next functions will:

1. correct the phone's stored rotation,
2. convert the photograph to grayscale,
3. turn dark marker strokes into bright pixels,
4. remove tiny isolated components,
5. crop around the drawing,
6. resize it while preserving its shape,
7. center it on a 28 × 28 black canvas,
8. normalize it for the CNN.

### Tips for a good drawing photograph

- Use a dark marker on plain white paper.
- Draw only one object.
- Fill a large part of the page.
- Avoid writing the category name.
- Use even lighting and avoid strong shadows.

In [ ]:
def center_by_mass(image):
    """Shift a white-on-black drawing toward the canvas center."""
    moments = cv2.moments(image)

    if moments['m00'] == 0:
        return image

    center_x = moments['m10'] / moments['m00']
    center_y = moments['m01'] / moments['m00']
    desired_center = (image.shape[0] - 1) / 2

    shift_x = int(round(desired_center - center_x))
    shift_y = int(round(desired_center - center_y))

    matrix = np.float32([
        [1, 0, shift_x],
        [0, 1, shift_y],
    ])

    return cv2.warpAffine(
        image,
        matrix,
        (image.shape[1], image.shape[0]),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )


def preprocess_drawing_photo(filename):
    """Convert a dark-on-white drawing photograph into a 28x28 doodle."""
    with Image.open(filename) as image:
        image = ImageOps.exif_transpose(image)
        original_rgb = np.asarray(
            image.convert('RGB')
        )
        grayscale = np.asarray(
            image.convert('L')
        )

    blurred = cv2.GaussianBlur(
        grayscale,
        (5, 5),
        0
    )

    _, binary = cv2.threshold(
        blurred,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    number_of_components, component_map, stats, _ = (
        cv2.connectedComponentsWithStats(
            binary,
            connectivity=8
        )
    )

    cleaned = np.zeros_like(binary)
    minimum_area = max(8, int(binary.size * 0.0005))

    for component_index in range(1, number_of_components):
        area = stats[
            component_index,
            cv2.CC_STAT_AREA
        ]

        if area >= minimum_area:
            cleaned[
                component_map == component_index
            ] = 255

    nonzero = cv2.findNonZero(cleaned)

    if nonzero is None:
        raise ValueError(
            'No dark drawing was detected. Use a darker marker '
            'and a plain, well-lit background.'
        )

    x, y, width, height = cv2.boundingRect(nonzero)

    padding = max(
        4,
        int(round(0.08 * max(width, height)))
    )

    left = max(0, x - padding)
    top = max(0, y - padding)
    right = min(cleaned.shape[1], x + width + padding)
    bottom = min(cleaned.shape[0], y + height + padding)

    cropped = cleaned[top:bottom, left:right]

    inner_size = 20
    scale = min(
        inner_size / cropped.shape[1],
        inner_size / cropped.shape[0]
    )

    resized_width = max(
        1,
        int(round(cropped.shape[1] * scale))
    )
    resized_height = max(
        1,
        int(round(cropped.shape[0] * scale))
    )

    resized = cv2.resize(
        cropped,
        (resized_width, resized_height),
        interpolation=cv2.INTER_AREA
    )

    canvas = np.zeros(
        (28, 28),
        dtype=np.uint8
    )

    x_start = (28 - resized_width) // 2
    y_start = (28 - resized_height) // 2

    canvas[
        y_start:y_start + resized_height,
        x_start:x_start + resized_width
    ] = resized

    processed_28x28 = center_by_mass(canvas)

    return {
        'original_rgb': original_rgb,
        'grayscale': grayscale,
        'binary': binary,
        'cleaned': cleaned,
        'cropped': cropped,
        'processed_28x28': processed_28x28,
    }


def show_preprocessing_stages(result):
    stage_names = [
        'Original photograph',
        'Grayscale',
        'Thresholded and inverted',
        'Cleaned',
        'Cropped drawing',
        'Final 28 × 28 input',
    ]

    stage_images = [
        result['original_rgb'],
        result['grayscale'],
        result['binary'],
        result['cleaned'],
        result['cropped'],
        result['processed_28x28'],
    ]

    plt.figure(figsize=(15, 8))

    for position, (name, image) in enumerate(
        zip(stage_names, stage_images),
        start=1
    ):
        plt.subplot(2, 3, position)

        if image.ndim == 2:
            plt.imshow(
                image,
                cmap='gray',
                vmin=0,
                vmax=255
            )
        else:
            plt.imshow(image)

        plt.title(name)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# STUDENT CUSTOMIZATION:
# Raise this threshold to make the system more cautious.

CONFIDENCE_THRESHOLD = 0.55


def predict_processed_drawing(processed_image, top_k=3):
    """Return the highest-probability guesses for one 28x28 drawing."""
    if model is None:
        print('Define and train the model first.')
        return None

    processed_image = np.asarray(processed_image)

    if processed_image.shape != (28, 28):
        raise ValueError(
            'processed_image must have shape (28, 28).'
        )

    model_input = (
        processed_image.astype('float32')
        / 255.0
    )
    model_input = model_input[
        np.newaxis,
        ...,
        np.newaxis,
    ]

    probabilities = model.predict(
        model_input,
        verbose=0
    )[0]

    top_indices = np.argsort(
        probabilities
    )[::-1][:top_k]

    top_guesses = [
        {
            'category': class_names[int(index)],
            'probability': float(probabilities[index]),
        }
        for index in top_indices
    ]

    return {
        'model_input': model_input,
        'top_guesses': top_guesses,
        'accepted': (
            top_guesses[0]['probability']
            >= CONFIDENCE_THRESHOLD
        ),
    }


def upload_and_predict_drawing():
    """Upload, preprocess, and classify one drawing photograph."""
    if model is None:
        print('Define and train the model first.')
        return None

    try:
        from google.colab import files
    except ModuleNotFoundError:
        print('The upload button in this cell is designed for Google Colab.')
        return None

    print('Upload one JPEG or PNG photograph of a drawing.')
    uploaded_files = files.upload()

    if not uploaded_files:
        print('No file was uploaded.')
        return None

    filename = next(iter(uploaded_files))

    # Colab normally saves the uploaded file automatically.
    # This fallback guarantees that it exists on disk.
    if not Path(filename).exists():
        Path(filename).write_bytes(
            uploaded_files[filename]
        )

    preprocessing = preprocess_drawing_photo(
        filename
    )
    show_preprocessing_stages(
        preprocessing
    )

    prediction = predict_processed_drawing(
        preprocessing['processed_28x28'],
        top_k=3
    )

    if prediction is None:
        return None

    print('\nTop guesses:')

    for rank, guess in enumerate(
        prediction['top_guesses'],
        start=1
    ):
        print(
            f"{rank}. {guess['category']}: "
            f"{100 * guess['probability']:.1f}%"
        )

    if prediction['accepted']:
        print(
            '\nThe top confidence passed the chosen threshold.'
        )
    else:
        print(
            '\nLow-confidence guess: the drawing may be unclear '
            'or unlike the training doodles.'
        )

    return {
        **prediction,
        'filename': filename,
        'preprocessing': preprocessing,
    }

In [ ]:
# Run this cell after training to test one drawing.

photo_prediction = upload_and_predict_drawing()

## Optional controlled experiment
Change exactly one item: filter count, convolution depth, augmentation, dropout, batch size, epoch count, confidence threshold, or scoring rule. Compare training, validation, test, Grad-CAM, and game behavior.